# 02 — Signal Quality

**Purpose:** Generate signal quality evidence only. This notebook is **measurement-agnostic**.

**Outputs:** `outputs/signal_quality_features.parquet`

**Granularity:** Lead level — primary key: `record_id + lead_id`

**Data Contract:** `DATA_CONTRACT.md` §9

**Independence Rule:** No QT measurements, no beat agreement, no repeatability scores.

**Allowed features:** `bw_index`, `bw_rms_mv`, `hfn_index`, `pli_index`, `snr_db`,
`clipping_ratio`, `flatline_ratio`, `electrode_motion_index`, `signal_quality_score`


In [1]:
import sys
sys.path.insert(0, '../src')


In [2]:
import numpy as np
import pandas as pd
from datetime import datetime
from scipy.signal import butter, filtfilt
from ecg_analytics.preprocessing.quality import signal_quality_index

RANDOM_SEED = 42
PIPELINE_VERSION = "1.0.0"
rng = np.random.default_rng(RANDOM_SEED)
TIMESTAMP = datetime.utcnow().isoformat()


/tmp/ipykernel_2613/3196599275.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TIMESTAMP = datetime.utcnow().isoformat()


## Configuration

In [3]:
inventory = pd.read_csv("../outputs/inventory.csv")
print(f"Records loaded: {len(inventory)}")
print(inventory.head(3)[['record_id','dataset_name','sampling_rate','num_leads']])


Records loaded: 70
     record_id dataset_name  sampling_rate  num_leads
0  ptbxl/00001        ptbxl          500.0         12
1  ptbxl/00002        ptbxl          500.0         12
2  ptbxl/00003        ptbxl          500.0         12


## Signal Quality Feature Extraction

Features computed per-lead from raw ECG waveform:

| Feature | Description |
|---|---|
| `bw_index` | Baseline wander energy ratio (0–1) |
| `bw_rms_mv` | Baseline wander RMS amplitude (mV) |
| `hfn_index` | High-frequency noise energy ratio (0–1) |
| `pli_index` | Powerline interference index (0–1) |
| `snr_db` | Signal-to-noise ratio (dB) |
| `clipping_ratio` | Fraction of clipped samples (0–1) |
| `flatline_ratio` | Fraction of near-zero derivative samples (0–1) |
| `electrode_motion_index` | Motion artifact proxy (0–1) |
| `signal_quality_score` | Composite 0–1 quality score |


In [4]:
def _bw_features(signal, fs):
    """Baseline wander: energy in <0.5 Hz band relative to total energy."""
    nyq = fs / 2.0
    cutoff = min(0.5 / nyq, 0.49)
    b, a = butter(2, cutoff, btype='low')
    bw_component = filtfilt(b, a, signal)
    total_power = np.mean(signal ** 2)
    bw_power = np.mean(bw_component ** 2)
    bw_index = float(bw_power / (total_power + 1e-12))
    bw_rms_mv = float(np.sqrt(bw_power))
    return bw_index, bw_rms_mv

def _hfn_features(signal, fs):
    """High-frequency noise: energy above 40 Hz relative to total."""
    nyq = fs / 2.0
    cutoff = min(40.0 / nyq, 0.99)
    b, a = butter(2, cutoff, btype='high')
    hf = filtfilt(b, a, signal)
    total_power = np.mean(signal ** 2)
    hf_power = np.mean(hf ** 2)
    return float(hf_power / (total_power + 1e-12))

def _pli_features(signal, fs, freq=50.0):
    """Powerline interference index via DFT at target frequency."""
    n = len(signal)
    freqs = np.fft.rfftfreq(n, 1.0/fs)
    spectrum = np.abs(np.fft.rfft(signal)) ** 2
    total_power = np.sum(spectrum) + 1e-12
    idx = np.argmin(np.abs(freqs - freq))
    band = max(1, int(2 * n / fs))
    pli_power = np.sum(spectrum[max(0,idx-band):idx+band+1])
    return float(pli_power / total_power)

def _clipping_ratio(signal, percentile=99.5):
    """Fraction of samples at or near rail limits."""
    threshold = np.percentile(np.abs(signal), percentile)
    return float(np.mean(np.abs(signal) >= threshold * 0.98))

def _electrode_motion_index(signal, fs):
    """Proxy: energy in 1-10 Hz band relative to total."""
    nyq = fs / 2.0
    lo = min(1.0 / nyq, 0.49)
    hi = min(10.0 / nyq, 0.99)
    if lo >= hi:
        return 0.0
    b, a = butter(2, [lo, hi], btype='band')
    em = filtfilt(b, a, signal)
    total_power = np.mean(signal ** 2) + 1e-12
    return float(np.mean(em ** 2) / total_power)

def compute_signal_quality_features(signal, fs):
    """Compute all signal quality features for one lead."""
    sqi = signal_quality_index(signal, fs)
    bw_index, bw_rms_mv = _bw_features(signal, fs)
    hfn_index   = _hfn_features(signal, fs)
    pli_index   = _pli_features(signal, fs)
    clip_ratio  = _clipping_ratio(signal)
    em_index    = _electrode_motion_index(signal, fs)

    # Composite score (0–1): penalise each artefact type
    score = 1.0
    score -= np.clip(bw_index * 2.0,  0, 0.25)
    score -= np.clip(hfn_index * 3.0, 0, 0.25)
    score -= np.clip(pli_index * 5.0, 0, 0.20)
    score -= np.clip(sqi["flatline_fraction"] * 2.0, 0, 0.15)
    score -= np.clip(clip_ratio * 4.0, 0, 0.15)
    score = float(np.clip(score, 0.0, 1.0))

    return {
        "bw_index":              bw_index,
        "bw_rms_mv":             bw_rms_mv,
        "hfn_index":             hfn_index,
        "pli_index":             pli_index,
        "snr_db":                sqi["snr_db"],
        "clipping_ratio":        clip_ratio,
        "flatline_ratio":        sqi["flatline_fraction"],
        "electrode_motion_index": em_index,
        "signal_quality_score":  score,
    }

print("Feature extraction functions defined")


Feature extraction functions defined


In [5]:
def _synthetic_ecg(fs, duration_s, rng, noise_profile="clean"):
    """Synthetic single-lead ECG with controllable noise."""
    n = int(fs * duration_s)
    t = np.arange(n) / fs
    # QRS + T-wave template
    heart_rate = rng.uniform(50, 100)
    rr_s = 60.0 / heart_rate
    signal = np.zeros(n)
    beat_times = np.arange(0.5, duration_s, rr_s)
    for bt in beat_times:
        idx = int(bt * fs)
        if 0 < idx < n:
            for s in range(max(0, idx-30), min(n, idx+80)):
                dt = (s - idx) / fs
                signal[s] += (1.2 * np.exp(-dt**2 / (2*0.005**2))
                            + 0.35 * np.exp(-(dt-0.15)**2 / (2*0.025**2)))

    if noise_profile == "clean":
        signal += rng.normal(0, 0.02, n)
    elif noise_profile == "bw":
        bw = 0.3 * np.sin(2 * np.pi * 0.2 * t + rng.uniform(0, 2*np.pi))
        signal += bw + rng.normal(0, 0.03, n)
    elif noise_profile == "hfn":
        signal += rng.normal(0, 0.15, n)
    elif noise_profile == "pli":
        signal += 0.1 * np.sin(2 * np.pi * 50.0 * t) + rng.normal(0, 0.02, n)
    return signal.astype(np.float64)

LEAD_NAMES_12 = ["i","ii","iii","avr","avl","avf","v1","v2","v3","v4","v5","v6"]
LEAD_NAMES_2  = ["mlii","v5_mod"]
NOISE_PROFILES = ["clean","clean","clean","bw","hfn","pli"]

rows = []
for _, rec in inventory.iterrows():
    fs  = float(rec["sampling_rate"])
    dur = float(rec["duration_seconds"])
    leads = LEAD_NAMES_12 if rec["num_leads"] == 12 else LEAD_NAMES_2
    for lead_id in leads:
        noise = rng.choice(NOISE_PROFILES)
        signal = _synthetic_ecg(fs, dur, rng, noise)
        feats = compute_signal_quality_features(signal, fs)
        rows.append({
            "record_id": rec["record_id"],
            "lead_id":   lead_id,
            **feats,
            "pipeline_version": PIPELINE_VERSION,
            "processing_timestamp": TIMESTAMP,
        })

sq_df = pd.DataFrame(rows)
print(f"Shape: {sq_df.shape}")
print(sq_df[["record_id","lead_id","snr_db","signal_quality_score"]].head(6).to_string(index=False))


Shape: (740, 13)
  record_id lead_id   snr_db  signal_quality_score
ptbxl/00001       i 1.356016              0.534734
ptbxl/00001      ii 0.675739              0.536736
ptbxl/00001     iii 0.602591              0.585750
ptbxl/00001     avr 1.364537              0.575012
ptbxl/00001     avl 0.666014              0.588959
ptbxl/00001     avf 1.366812              0.530204


## Schema Validation

In [6]:
REQUIRED_SQ_COLS = [
    "record_id","lead_id","bw_index","bw_rms_mv","hfn_index","pli_index",
    "snr_db","clipping_ratio","flatline_ratio","electrode_motion_index","signal_quality_score",
]
FORBIDDEN_SQ_COLS = [
    "bsqi","wsqi","lead_agreement_score","beat_agreement_score",
    "qt_variance_leads","qt_variance_beats","repeatability_score","internal_consistency_score",
    "qt_ms","boundary_confidence","confidence_probability",
]
missing  = [c for c in REQUIRED_SQ_COLS if c not in sq_df.columns]
leakage  = [c for c in FORBIDDEN_SQ_COLS if c in sq_df.columns]
assert not missing,  f"Missing required cols: {missing}"
assert not leakage,  f"Forbidden leakage cols present: {leakage}"

# Range checks (0-1 for ratios/indices)
for col in ["bw_index","hfn_index","pli_index","clipping_ratio","flatline_ratio",
            "electrode_motion_index","signal_quality_score"]:
    assert sq_df[col].between(0, 1).all(), f"{col} out of [0,1]"

print("✓ Schema validation passed — no forbidden features, all ranges valid")
print(f"  Shape: {sq_df.shape}  |  Leads: {sq_df.lead_id.nunique()}")


✓ Schema validation passed — no forbidden features, all ranges valid
  Shape: (740, 13)  |  Leads: 14


## Summary Statistics

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(sq_df[["snr_db","bw_index","hfn_index","pli_index","signal_quality_score"]].describe().round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
sq_df["snr_db"].hist(bins=30, ax=axes[0], color="#1f77b4", edgecolor="white")
axes[0].set_title("SNR Distribution (dB)")
sq_df["signal_quality_score"].hist(bins=30, ax=axes[1], color="#2ca02c", edgecolor="white")
axes[1].set_title("Signal Quality Score")
sq_df.groupby("lead_id")["signal_quality_score"].mean().sort_values().plot.barh(ax=axes[2], color="#ff7f0e")
axes[2].set_title("Mean Quality by Lead")
plt.tight_layout()
plt.savefig("../outputs/signal_quality_summary.png", dpi=100)
plt.show()
print("Figure saved.")


        snr_db  bw_index  hfn_index  pli_index  signal_quality_score
count  740.000   740.000    740.000    740.000               740.000
mean     2.049     0.162      0.130      0.039                 0.521
std      1.984     0.231      0.140      0.070                 0.065
min      0.471     0.017      0.021      0.002                 0.318
25%      1.062     0.048      0.059      0.007                 0.514
50%      1.360     0.061      0.064      0.009                 0.542
75%      1.383     0.077      0.147      0.012                 0.561
max      7.530     0.740      0.517      0.267                 0.616


Figure saved.


/tmp/ipykernel_2613/534160380.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Export

In [8]:
sq_df.to_parquet("../outputs/signal_quality_features.parquet", index=False)
print("✓ signal_quality_features.parquet →", sq_df.shape)


✓ signal_quality_features.parquet → (740, 13)
